# 📊 Italienation – Comprehensive Data Inventory

**Purpose:** Full audit of every dataset, notebook, and processed output in the project. Identifies what we have, what's missing, and the coverage/quality of each data domain.

**Project scope:** Comparative study of Italian NEETs (Not in Education, Employment, or Training), education system quality, fiscal landscape, labour-market transitions, and broader human-capital indicators.

---

In [1]:
import os, json, glob
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML

ROOT = Path(r'c:\Users\Dell\Documents\Antigravity\Italienation')
LOCAL = ROOT / 'local_data'
PROCESSED = LOCAL / 'processed'
NOTEBOOKS = ROOT / 'Notebooks'

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 200)

def sizeof_fmt(num):
    for unit in ['B','KB','MB','GB']:
        if abs(num) < 1024.0:
            return f'{num:3.1f} {unit}'
        num /= 1024.0
    return f'{num:.1f} TB'

def scan_dir(path, extensions=None):
    """Return a DataFrame of files under `path`."""
    rows = []
    for f in sorted(Path(path).rglob('*')):
        if f.is_file():
            if extensions and f.suffix.lower() not in extensions:
                continue
            rows.append({
                'file': str(f.relative_to(ROOT)),
                'name': f.name,
                'ext': f.suffix.lower(),
                'size_bytes': f.stat().st_size,
                'size_human': sizeof_fmt(f.stat().st_size),
                'parent': str(f.parent.relative_to(ROOT)),
            })
    return pd.DataFrame(rows)

print('Root:', ROOT)
print('local_data exists:', LOCAL.exists())
print('processed exists:', PROCESSED.exists())

Root: c:\Users\Dell\Documents\Antigravity\Italienation
local_data exists: True
processed exists: True


---
## 1. High-Level File Census

Count every file under `local_data/` by source folder and file extension.

In [2]:
all_files = scan_dir(LOCAL)
print(f'Total files under local_data/: {len(all_files)}')
print(f'Total size: {sizeof_fmt(all_files["size_bytes"].sum())}\n')

# Top-level source folders
all_files['source'] = all_files['file'].apply(lambda x: x.split(os.sep)[1] if len(x.split(os.sep)) > 1 else 'root')
source_summary = all_files.groupby('source').agg(
    file_count=('name', 'count'),
    total_size_bytes=('size_bytes', 'sum'),
).sort_values('file_count', ascending=False)
source_summary['total_size'] = source_summary['total_size_bytes'].apply(sizeof_fmt)
display(Markdown('### Files by Source'))
display(source_summary[['file_count', 'total_size']])

Total files under local_data/: 691
Total size: 3.6 GB



### Files by Source

,file_count,total_size
source,,
MUR,132,245.6 MB
processed,86,13.6 MB
eurostat,74,576.9 MB
ISTAT,71,113.3 MB
INVALSI,49,180.0 MB
MinIstruzione,48,1019.8 MB
UKSDGstats,48,19.9 MB
ourWorldData,43,16.3 MB
oecd,24,113.1 MB


In [3]:
# By extension
ext_summary = all_files.groupby('ext').agg(
    count=('name', 'count'),
    total_bytes=('size_bytes', 'sum'),
).sort_values('count', ascending=False)
ext_summary['total_size'] = ext_summary['total_bytes'].apply(sizeof_fmt)
display(Markdown('### Files by Extension'))
display(ext_summary[['count', 'total_size']])

### Files by Extension

,count,total_size
ext,,
.csv,540,3.5 GB
.json,42,327.0 KB
.md,36,209.7 KB
.xlsx,25,2.5 MB
.tsv,18,1.1 MB
.xml,9,15.8 MB
.html,7,1.1 MB
.zip,6,95.4 MB
.pdf,4,8.4 MB


---
## 2. Source-by-Source Data Inventory

### 2.1 ISTAT (Istituto Nazionale di Statistica)

Italy's national statistics office. Core source for NEETs, labour force, poverty, education, school data.

In [4]:
istat_files = all_files[all_files['source'] == 'ISTAT'].copy()
display(Markdown(f'**{len(istat_files)} files** under `local_data/ISTAT/`'))

# Group by thematic sub-area
def istat_theme(name):
    n = name.lower()
    if 'neet' in n: return '🔴 NEET'
    if 'poverty' in n or 'poverta' in n: return '💰 Poverty'
    if 'household' in n: return '🏠 Household spending'
    if 'school' in n or 'scuola' in n or 'infanzia' in n or 'primaria' in n or 'sec1' in n or 'sec2' in n: return '🏫 Schools (enrolment)'
    if 'universit' in n or 'laureati' in n: return '🎓 University/Graduates'
    if 'diplom' in n or 'highschool' in n: return '📜 HS graduates/diplomas'
    if 'employment' in n or 'occupation' in n or 'labour' in n or 'disoccup' in n or 'tasso_occ' in n or 'unemployment' in n: return '💼 Labour market'
    if 'phd' in n or 'wages' in n: return '🧑‍🔬 PhD / wages'
    if 'disability' in n or 'mental' in n: return '♿ Disability/Health'
    if 'early_school' in n: return '🚪 Early school leaving'
    if 'manifest' in n: return '📋 Manifest'
    return '📁 Other'

istat_files['theme'] = istat_files['name'].apply(istat_theme)
display(istat_files.groupby('theme').agg(count=('name','count'), total=('size_bytes','sum')).sort_values('count', ascending=False))
print()
display(Markdown('**Full file list:**'))
for _, r in istat_files.iterrows():
    print(f"  {r['theme']:30s}  {r['size_human']:>10s}  {r['name']}")

**71 files** under `local_data/ISTAT/`

,count,total
theme,,
🎓 University/Graduates,15,2032937
🏫 Schools (enrolment),13,44908048
💰 Poverty,10,427355
📁 Other,9,30596658
🏠 Household spending,7,9997969
💼 Labour market,6,21881016
📋 Manifest,4,11184
🔴 NEET,3,7453548
📜 HS graduates/diplomas,2,314489


**Full file list:**

  📋 Manifest                          1.6 KB  manifest.json
  📁 Other                            66.0 KB  istat_tertiary_degree_employees_by_nace2.csv
  🎓 University/Graduates             16.9 KB  istat_university_graduates_job_search.csv
  🎓 University/Graduates             41.4 KB  istat_university_graduates_monthly_income.csv
  🎓 University/Graduates             37.3 KB  istat_university_graduates_occupational_condition.csv
  🎓 University/Graduates             35.6 KB  istat_university_graduates_profession_group.csv
  📋 Manifest                          5.9 KB  manifest.json
  📁 Other                           267.5 KB  istat_tertiary_degree_employees_by_nace2.xml
  🎓 University/Graduates             90.0 KB  istat_university_graduates_job_search.xml
  🎓 University/Graduates            304.4 KB  istat_university_graduates_monthly_income.xml
  🎓 University/Graduates            234.7 KB  istat_university_graduates_occupational_condition.xml
  🎓 University/Graduates            246.1 KB

In [5]:
# Peek at key ISTAT CSV shapes
istat_csvs = istat_files[istat_files['ext'] == '.csv']
peek_rows = []
for _, r in istat_csvs.head(20).iterrows():
    fpath = ROOT / r['file']
    try:
        tmp = pd.read_csv(fpath, nrows=0, encoding='utf-8')
    except:
        try:
            tmp = pd.read_csv(fpath, nrows=0, encoding='latin1')
        except:
            tmp = pd.DataFrame()
    nrows = sum(1 for _ in open(fpath, encoding='utf-8', errors='ignore')) - 1
    peek_rows.append({
        'file': r['name'],
        'columns': len(tmp.columns),
        'col_names': ', '.join(tmp.columns[:8]),
        'rows_approx': nrows,
        'size': r['size_human'],
    })
display(Markdown('### ISTAT CSV Shapes (first 20)'))
display(pd.DataFrame(peek_rows))

### ISTAT CSV Shapes (first 20)

,file,columns,col_names,rows_approx,size
0,istat_tertiary_degree_employees_by_nace2.csv,16,"freq, ref_area, data_type, econ_activity_nace_2007, pers_empl_size_class, info_innov_enterprises, enterprise_goals, ...",392,66.0 KB
1,istat_university_graduates_job_search.csv,16,"freq, ref_area, data_type, sex, type_of_degree, posiz_prof_desired, full_part_time_pref, place_of_work",126,16.9 KB
2,istat_university_graduates_monthly_income.csv,20,"freq, ref_area, data_type, sex, type_of_degree, field_study, labprof_status_e, labprof_status_d",306,41.4 KB
3,istat_university_graduates_occupational_condition.csv,20,"freq, ref_area, data_type, sex, type_of_degree, field_study, labprof_status_e, labprof_status_d",251,37.3 KB
4,istat_university_graduates_profession_group.csv,20,"freq, ref_area, data_type, sex, type_of_degree, field_study, labprof_status_e, labprof_status_d",255,35.6 KB
5,istat_diplomati_caratteristiche.csv,26,"DATAFLOW, FREQ, ETA_INTERVISTA, CORSI_FORMPROF_DIPL, ISTRUZ_TERZ_NONUNIV, ISTRUZ_UNIV, ITTER107, NATGIUR_SCUOLA",3923,186.2 KB
6,istat_diplomati_occupazione_retribuzione.csv,25,"DATAFLOW, FREQ, ATT_ECO, CONDIZIONE_DICH3_OCC, PROFESSIONE_CP2011, ITTER107, ORE_SETT_LAVORO, TIPO_OCC",2925,120.9 KB
7,istat_early_school_leavers.csv,18,"DATAFLOW, FREQ, CITTADINANZA, CONDIZIONE_PROF, ITTER107, SESSO, TIPO_DATO, TIME_PERIOD",4619,120.7 KB
8,istat_graduates_job_search.csv,13,"freq, itter107, occupaz_desid, luogo_lavoro, regime_orario_des, sesso, ricerca_lavoro, tipo_dato",420,20.6 KB
9,istat_graduates_occupational_status.csv,17,"freq, gruppo_laurea, att_eco, condizione_dich3_occ, anno_conseg_laurea, itter107, isco3d, tipo_occ",3349,217.3 KB


### 2.2 Eurostat

European-level NEET, education spending, unemployment, wellbeing, VET, housing, and more.

In [6]:
estat_files = all_files[all_files['source'] == 'eurostat'].copy()
display(Markdown(f'**{len(estat_files)} files** under `local_data/eurostat/`'))

def estat_theme(name):
    n = name.lower()
    if 'neet' in n: return '🔴 NEET'
    if 'unemployment' in n: return '💼 Unemployment'
    if 'educ_expenditure' in n or 'spending' in n: return '💰 Education spending'
    if 'training' in n: return '📚 Training/lifelong'
    if 'esl' in n or 'early_school' in n: return '🚪 Early school leaving'
    if 'tertiary' in n: return '🎓 Tertiary education'
    if 'upper_secondary' in n or 'graduation' in n: return '📜 Upper secondary'
    if 'vet' in n: return '🔧 VET (vocational)'
    if 'poverty' in n or 'homeownership' in n or 'housing' in n: return '🏠 Poverty/Housing'
    if 'health' in n or 'wellbeing' in n or 'mental' in n: return '🧠 Health/Wellbeing'
    if 'trust' in n or 'satisfaction' in n or 'social' in n: return '🤝 Social/Trust'
    if 'labour' in n or 'contract' in n or 'overqual' in n or 'vacancy' in n or 'long_term' in n: return '💼 Labour quality'
    if 'broadband' in n or 'transport' in n or 'crime' in n: return '🏗️ Infrastructure'
    if 'fertility' in n: return '👶 Fertility'
    if 'migration' in n: return '🌍 Migration'
    if 'gdp' in n or 'regional' in n: return '📊 Regional GDP'
    if 'grade_retention' in n: return '📖 Grade retention'
    if 'adult_learning' in n: return '📚 Adult learning'
    return '📁 Other'

estat_files['theme'] = estat_files['name'].apply(estat_theme)
display(estat_files.groupby('theme').agg(count=('name','count'), total_MB=('size_bytes', lambda x: round(x.sum()/1e6,1))).sort_values('count', ascending=False))
print()
print('Top 10 largest Eurostat files:')
for _, r in estat_files.nlargest(10, 'size_bytes').iterrows():
    print(f"  {r['size_human']:>10s}  {r['name']}")

**74 files** under `local_data/eurostat/`

,count,total_MB
theme,,
📁 Other,20,39.3
🔴 NEET,8,44.1
💼 Unemployment,6,4.5
💼 Labour quality,6,39.8
📚 Training/lifelong,4,2.2
🏗️ Infrastructure,4,7.8
🧠 Health/Wellbeing,4,135.7
🎓 Tertiary education,3,26.8
🔧 VET (vocational),3,12.9



Top 10 largest Eurostat files:
    163.5 MB  eurostat_it_upper_secondary_graduation_by_programme.csv
    116.8 MB  eurostat_self_reported_health.csv
     25.1 MB  eurostat_tertiary_field_distribution.csv
     23.5 MB  estat_young_not_in_edu_employment_by_education_years_since_completion.csv
     20.3 MB  eurostat_it_neet_by_education_level.csv
     18.6 MB  eurostat_it_home_ownership_housing_cost_proxy.csv
     18.5 MB  eurostat_job_vacancy_rate.csv
     17.7 MB  eurostat_esl_by_migration.csv
     16.6 MB  eurostat_homeownership_by_age.csv
     16.0 MB  eurostat_it_overqualification_proxy.csv


### 2.3 OECD

Education spending, teacher experience, PISA, VET, transition data, earnings, and minimum-wage metrics.

In [7]:
oecd_files = all_files[all_files['source'] == 'oecd'].copy()
display(Markdown(f'**{len(oecd_files)} files** under `local_data/oecd/`'))
for _, r in oecd_files.iterrows():
    print(f"  {r['size_human']:>10s}  {r['name']}")

**24 files** under `local_data/oecd/`

    493.3 KB  educ_uoe_fini01$defaultview_linear_2_0.csv
     39.3 KB  eurostat_gdp_per_capita.csv
     14.4 MB  OECD.ELS.JAI,DSD_TAXBEN_HOURSPOV@DF_HOURSPOV,1.0+all.csv
      2.0 MB  OECD.ELS.SAE,DSD_EARNINGS@AGE_WAGE_GAP,1.0+all.csv
      1.2 MB  OECD.ELS.SAE,DSD_EARNINGS@PAY_INCIDENCE,1.0+all.csv
      1.6 MB  OECD.ELS.SAE,DSD_EARNINGS@RMW,1.0+all.csv
     98.9 KB  OECD.GOV.GIP,DSD_GOV@DF_GOV_INFPD_2025,1.0+all.csv
    822.2 KB  OECD.GOV.GIP,DSD_GOV@DF_GOV_PPROC_2025,1.0+all.csv
      5.5 MB  OECD.GOV.GIP,DSD_QDD_GOV_PUBPRO_2024@DF_GOV_PUBPRO_2024,1.0+all.csv
      9.4 MB  OECD.SDD.TPS,DSD_EAR@DF_HOU_EAR,1.0+all.csv
     42.6 MB  oecd_eag_transition.csv
     35.5 KB  oecd_education_attainment_migration.csv
    348.3 KB  oecd_education_costs.csv
    158.1 KB  oecd_education_fin_gdp.csv
     20.9 KB  oecd_education_fin_indic_source_nature.csv
    140.8 KB  oecd_education_fin_perstud.csv
    652.8 KB  oecd_education_funding_sources.csv
    114.6 KB  oecd_education_nature_cur_cap.csv
  

### 2.4 World Bank

In [8]:
wb_files = all_files[all_files['source'] == 'worldbank'].copy()
display(Markdown(f'**{len(wb_files)} files** under `local_data/worldbank/`'))
for _, r in wb_files.iterrows():
    print(f"  {r['size_human']:>10s}  {r['name']}")

# Also root-level WB files
root_wb = all_files[all_files['name'].str.startswith('WB_')]
if len(root_wb):
    print('\nRoot-level World Bank files:')
    for _, r in root_wb.iterrows():
        print(f"  {r['size_human']:>10s}  {r['name']}")

**7 files** under `local_data/worldbank/`

      2.6 MB  wb_education_spending_pct_gdp.csv
      3.8 MB  wb_learning_poverty.csv
    171.6 KB  wb_suicide_mortality.csv
    119.2 KB  wb_teachers_trained_primary.csv
     89.6 KB  wb_teachers_trained_secondary.csv
      2.3 MB  wb_tertiary_enrollment_gross.csv
      2.8 MB  wb_tertiary_spending_pct_gdp_percapita.csv

Root-level World Bank files:
    950.0 KB  WB_WDI_SI_POV_GINI.csv


### 2.5 INVALSI (National Student Assessment)

Standardised test scores, dispersion, implicit dropout, digital competence.

In [9]:
invalsi_files = all_files[all_files['source'] == 'INVALSI'].copy()
display(Markdown(f'**{len(invalsi_files)} files** under `local_data/INVALSI/`'))
print(f'Total size: {sizeof_fmt(invalsi_files["size_bytes"].sum())}\n')

# Group by school year / report type
def invalsi_year(name):
    for y in ['2024-2025', '2023-2024', '2022-2023']:
        if y in name: return y
    if 'eccellenza' in name.lower() or 'dispersione' in name.lower(): return 'Dispersione/Eccellenza'
    if 'punteggi' in name.lower(): return 'Punteggi (scores)'
    if 'manifest' in name.lower(): return 'Manifest'
    return 'Other'

invalsi_files['report_year'] = invalsi_files['name'].apply(invalsi_year)
display(invalsi_files.groupby('report_year').agg(count=('name','count')).sort_index())

**49 files** under `local_data/INVALSI/`

Total size: 180.0 MB



,count
report_year,
2022-2023,5
2023-2024,4
2024-2025,32
Dispersione/Eccellenza,3
Manifest,1
Other,1
Punteggi (scores),3


### 2.6 Ministero dell'Istruzione (MinIstruzione)

School-level open data: students (Alunni), teachers (Docenti), school buildings (Edifici), textbooks (LibriDiTesto), school registry (Scuole), budgets (BilancioeFinanze).

In [10]:
mini_files = all_files[all_files['source'] == 'MinIstruzione'].copy()
display(Markdown(f'**{len(mini_files)} files** under `local_data/MinIstruzione/`'))
print(f'Total size: {sizeof_fmt(mini_files["size_bytes"].sum())}\n')

def mini_area(path):
    parts = path.split(os.sep)
    if len(parts) >= 3: return parts[2]
    return 'root'

mini_files['area'] = mini_files['file'].apply(mini_area)
display(mini_files.groupby('area').agg(
    files=('name','count'),
    size_MB=('size_bytes', lambda x: round(x.sum()/1e6,1))
))

**48 files** under `local_data/MinIstruzione/`

Total size: 1019.8 MB



,files,size_MB
area,,
Alunni,8,38.5
BilancioeFinanze,7,300.0
Docenti,1,0.2
Edifici,9,37.2
LibriDiTesto,17,662.2
Scuole,6,31.2


### 2.7 MUR (Ministero dell'Università e della Ricerca)

University graduates, enrollment, AFAM, DSU (right-to-study), university staff.

In [11]:
mur_files = all_files[all_files['source'] == 'MUR'].copy()
display(Markdown(f'**{len(mur_files)} files** under `local_data/MUR/`'))
print(f'Total size: {sizeof_fmt(mur_files["size_bytes"].sum())}\n')

def mur_area(path):
    parts = path.split(os.sep)
    if len(parts) >= 3: return parts[2]
    return 'root'

mur_files['area'] = mur_files['file'].apply(mur_area)
display(mur_files.groupby('area').agg(
    files=('name','count'),
    size_MB=('size_bytes', lambda x: round(x.sum()/1e6,1))
))

**132 files** under `local_data/MUR/`

Total size: 245.6 MB



,files,size_MB
area,,
2024-contribuzione-e-interventi-atenei,8,1.4
2025-collegi-universitari,7,0.2
2025-contribuzione-e-interventi-afam,5,1.7
2025-diritto-allo-studio-universitario-dsu-regionale,7,3.1
MUR_iscritti,18,8.1
atenei.csv,1,0.0
classidilaurea.csv,1,0.1
dati-per-bilancio-di-genere,21,44.0
diplomati-afam-serie-storica,7,17.0


### 2.8 SIOPE (School-level Treasury Flows)

In [12]:
siope_files = all_files[all_files['source'] == 'SIOPE'].copy()
display(Markdown(f'**{len(siope_files)} files** under `local_data/SIOPE/`'))
for _, r in siope_files.sort_values('name').iterrows():
    print(f"  {r['size_human']:>10s}  {r['name']}")

# Note empty years
print('\n⚠️ Note: siope_uscite_2014 through _2019 are 59 bytes each (header-only / empty).')

**9 files** under `local_data/SIOPE/`

     558.0 B  siope_2014_2019_resolution.md
    225.5 KB  siope_anagrafiche_scuole.csv
    386.7 KB  siope_uscite_2020.csv
      1.1 MB  siope_uscite_2021.csv
      1.2 MB  siope_uscite_2022.csv
      2.5 MB  siope_uscite_2023.csv
      7.4 MB  siope_uscite_2024.csv
     13.8 MB  siope_uscite_2025.csv
      4.0 MB  siope_uscite_2026.csv

⚠️ Note: siope_uscite_2014 through _2019 are 59 bytes each (header-only / empty).


### 2.9 MEF (IRPEF Fiscal Data)

In [13]:
mef_files = all_files[all_files['source'] == 'MEF'].copy()
display(Markdown(f'**{len(mef_files)} files** under `local_data/MEF/`'))
for _, r in mef_files.iterrows():
    print(f"  {r['size_human']:>10s}  {r['name']}")

**9 files** under `local_data/MEF/`

      3.5 KB  manifest.json
      2.2 MB  Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2024.csv
   1004.6 KB  mef_comunale_irpef_2024.zip
    190.3 KB  mef_reg_calcolo_irpef_2025.csv
    186.6 KB  mef_reg_tipo_reddito_2025.csv
     21.6 KB  mef_sesso_calcolo_irpef_2025.csv
     23.7 KB  mef_sesso_tipo_reddito_2025.csv
    199.0 KB  Redditi_e_principali_variabili_IRPEF_su_base_subcomunale_CSV_2024.csv
     87.4 KB  mef_subcomunale_irpef_2024.zip


### 2.10 Other Sources (ANPAL, INPS, AlmaLaurea, Our World in Data, OpenCoesione, UK SDGs)

In [14]:
other_sources = ['ANPAL', 'INPS', 'AlmaLaurea', 'ourWorldData', 'OpenCoesione', 'UKSDGstats', 'manual_required']
for src in other_sources:
    sf = all_files[all_files['source'] == src]
    print(f'\n--- {src} ({len(sf)} files, {sizeof_fmt(sf["size_bytes"].sum())}) ---')
    for _, r in sf.head(10).iterrows():
        print(f"  {r['size_human']:>10s}  {r['name']}")
    if len(sf) > 10:
        print(f'  ... and {len(sf)-10} more files')


--- ANPAL (6 files, 8.5 MB) ---
     450.0 B  ANPAL_DATA_STATUS.md
     136.0 B  anpal_replacement_early_school_leavers.csv
      2.0 KB  anpal_replacement_manifest.json
     964.0 B  anpal_replacement_neet_annual.csv
      8.4 MB  anpal_replacement_neet_by_migration.csv
     133.0 B  anpal_replacement_youth_unemployment.csv

--- INPS (21 files, 37.2 KB) ---
      1.2 KB  attivit_-ispettiva-di-vigilanza-per-ente-controllore_-aziende-ispezionate-e-lavoratori-non-regolari---anno-2008-2009-e-1_-semestre-2010__1.csv
      1.2 KB  attivit_-ispettiva-di-vigilanza-per-ente-controllore_-aziende-ispezionate-e-lavoratori-non__1.csv
      1.1 KB  attivit_-ispettiva-di-vigilanza-per-ente-controllore_aziende-ispezionate-e-lavoratori-non-regolari---anni-2010-2011-e-2012__1.csv
      3.6 KB  bambini-dai-3-anni-et_-obbligo-con-assistenza-formale-e-informale-e-accuditi-in-via-esclusiva-dai-genitori-nei-paesi-dell_ue-anni-2005-2011.iii.4.2.4__1.csv
     251.0 B  ID-2324.csv
     214.0 B  ID-2326.csv
  

### 2.11 Root-level local_data files (miscellaneous)

In [15]:
# Files directly under local_data/ (not in subdirectories)
root_files = all_files[~all_files['source'].isin(
    ['ISTAT','eurostat','oecd','worldbank','INVALSI','MinIstruzione','MUR','SIOPE','MEF',
     'ANPAL','INPS','AlmaLaurea','ourWorldData','OpenCoesione','UKSDGstats','manual_required','processed']
)]
if len(root_files):
    display(Markdown(f'**{len(root_files)} miscellaneous files** at local_data root:'))
    for _, r in root_files.iterrows():
        print(f"  {r['size_human']:>10s}  {r['name']}")

**16 miscellaneous files** at local_data root:

     31.1 KB  API_HD.HCI.OVRL_DS63_en_csv_v2_756596.csv
    493.3 KB  educ_uoe_fini01$defaultview_linear_2_0.csv
    397.4 KB  ESTAT_EDAT_LFSE_22$DEFAULTVIEW_1.0.xml
     84.1 KB  ESTAT_TPS00203_1.0.xml
      1.3 KB  Figure_1__The_percentage_of_young_people_who_are_not_in_education,_employment_or_training_(NEET)_increased_over_the_quarter_(January_to_March_2025).csv
    340.0 KB  Incidenza dei giovani Neet - Titolo di studio (_) (IT1,172_931_DF_DCCV_NEET1_8,1.0).csv
     774.0 B  ItalianMeanSecondarySchoolExpenses.csv
     734.0 B  ItalyPrimarySchoolBookExpenses.csv
     52.0 KB  Metadata_Country_API_HD.HCI.OVRL_DS63_en_csv_v2_756596.csv
     543.0 B  Metadata_Indicator_API_HD.HCI.OVRL_DS63_en_csv_v2_756596.csv
     67.8 KB  NEET  (giovani non occupati e non in istruzione e formazione) - Dati regionali (IT1,172_931_DF_DCCV_NEET1_6,1.0).csv
    686.2 KB  NEET  (giovani non occupati e non in istruzione e formazione)- Condizione professionele europea, cittadinanza (IT1,172_931_DF_DCCV_NEE

---
## 3. Processed / Derived Datasets

These are analysis-ready outputs under `local_data/processed/`. This is where the real analytical value lives.

In [16]:
proc_files = all_files[all_files['source'] == 'processed'].copy()
display(Markdown(f'**{len(proc_files)} processed files**'))

def proc_theme(name):
    n = name.lower()
    if 'neet' in n: return '🔴 NEET'
    if 'anpal' in n: return '🏛️ ANPAL replacement'
    if 'siope' in n: return '💰 SIOPE fiscal'
    if 'education_expenditure' in n or 'education_fiscal' in n or 'education_finance' in n: return '📊 Education spending'
    if 'transition' in n: return '🔄 School transitions'
    if 'istat_repeaters' in n or 'istat_lower' in n or 'school_outcome' in n or 'snv_esiti' in n: return '📖 School outcomes'
    if 'disability' in n or 'bes_' in n: return '♿ Disability/BES'
    if 'dsu' in n or 'ersu' in n or 'mur_' in n or 'atenei' in n: return '🎓 University support'
    if 'inps' in n: return '📋 INPS'
    if 'global' in n or 'italy_position' in n or 'worldbank' in n: return '🌍 International position'
    if 'household' in n or 'school_household' in n: return '🏠 Household costs'
    if 'save_the_children' in n: return '👶 Save the Children'
    if 'oed_' in n or 'oed_destination' in n: return '🎯 OED destination'
    if 'manifest' in n or 'source' in n: return '📋 Manifests/Sources'
    if 'policy' in n: return '📜 Policy benchmark'
    if 'ministry_students' in n: return '🏫 Ministry students'
    return '📁 Other'

proc_files['theme'] = proc_files['name'].apply(proc_theme)

for theme in sorted(proc_files['theme'].unique()):
    subset = proc_files[proc_files['theme'] == theme]
    print(f'\n{theme} ({len(subset)} files)')
    for _, r in subset.iterrows():
        print(f"  {r['size_human']:>10s}  {r['name']}")

**86 processed files**


♿ Disability/BES (5 files)
      1.9 KB  bes_disability_sources.md
      2.3 KB  bes_disability_sources_manifest.json
      4.7 KB  estimated_bes_students_by_region_order_2024_25_using_istat_rates.csv
      3.0 KB  istat_bes_rate_by_region_order_2022_2023.csv
     506.0 B  istat_disability_rate_timeseries_by_order.csv

🌍 International position (11 files)
     12.6 KB  global_he_cost_access_latest_year.csv
      1.9 MB  global_he_cost_access_panel.csv
      2.0 KB  global_he_cost_access_sources.md
      2.6 KB  global_he_cost_access_sources_manifest.json
     811.0 B  global_italy_position_method_notes.md
     76.5 KB  global_italy_position_oecd_wb_latest.csv
      2.8 KB  global_policy_response_benchmark_schema.md
      5.5 KB  global_policy_response_benchmark_sources_manifest.json
      1.2 KB  global_policy_response_benchmark_template.csv
     864.0 B  italy_position_summary_oecd_wb.csv
      1.5 KB  worldbank_italy_cpi_index.csv

🎓 University support (5 files)
     15.2 KB  atenei_

      1.1 KB  neet_regional_risk_model_coefficients.csv
     739.0 B  neet_regional_risk_model_metrics.json
      2.1 KB  neet_regional_risk_model_predictions.csv
      3.6 KB  neet_regional_target_panel.csv


In [17]:
# Peek into key processed CSVs
key_processed = [
    'neet_regional_model_panel.csv',
    'neet_gender_year_panel.csv',
    'education_expenditure_state_parents_gdp.csv',
    'education_fiscal_inventory.csv',
    'transition_bridge_model_panel.csv',
    'istat_repeaters_upper_secondary_latest.csv',
    'snv_esiti_school_year_proxy.csv',
    'global_italy_position_oecd_wb_latest.csv',
    'dsu_ersu_support_panel_2024_2025.csv',
]

for fname in key_processed:
    fpath = PROCESSED / fname
    if fpath.exists():
        try:
            df = pd.read_csv(fpath, nrows=5)
            full = pd.read_csv(fpath)
            print(f'\n📄 {fname} ({len(full)} rows × {len(full.columns)} cols)')
            print(f'   Columns: {list(full.columns)}')
            if 'year' in [c.lower() for c in full.columns]:
                yr_col = [c for c in full.columns if c.lower() == 'year'][0]
                print(f'   Year range: {full[yr_col].min()} – {full[yr_col].max()}')
            display(df.head(3))
        except Exception as e:
            print(f'\n⚠️ {fname}: {e}')
    else:
        print(f'\n❌ {fname}: NOT FOUND')


📄 neet_regional_model_panel.csv (198 rows × 23 cols)
   Columns: ['REF_AREA', 'REF_AREA_LABEL', 'TIME_PERIOD', 'lower_disability_per_1000_t_minus_1', 'lower_class_size_t_minus_1', 'lower_exam_success_t_minus_1', 'lower_foreign_share_t_minus_1', 'lower_median_grade_t_minus_1', 'lower_public_share_t_minus_1', 'lower_exam_failure_t_minus_1', 'upper_repeaters_all_t', 'upper_repeaters_fir_t', 'upper_lic_t', 'upper_tec_t', 'upper_voc_t', 'upper_voc_minus_lic_t', 'upper_tec_minus_lic_t', 'transition_jump_all_t', 'transition_jump_fir_t', 'neet_count_15_29', 'neet_risk_index', 'neet_percentile', 'covid_period']


,REF_AREA,REF_AREA_LABEL,TIME_PERIOD,lower_disability_per_1000_t_minus_1,lower_class_size_t_minus_1,lower_exam_success_t_minus_1,lower_foreign_share_t_minus_1,lower_median_grade_t_minus_1,lower_public_share_t_minus_1,lower_exam_failure_t_minus_1,...,upper_tec_t,upper_voc_t,upper_voc_minus_lic_t,upper_tec_minus_lic_t,transition_jump_all_t,transition_jump_fir_t,neet_count_15_29,neet_risk_index,neet_percentile,covid_period
0,ITC1,Piemonte,2016,37.9,21.1,99.7,12.9,7.0,95.3,0.3,...,NaN,NaN,NaN,NaN,6.8,11.5,NaN,NaN,NaN,NaN
1,ITC2,Valle d'Aosta / Vallée d'Aoste,2016,32.5,20.4,99.8,8.6,7.0,95.9,0.2,...,NaN,NaN,NaN,NaN,5.7,9.4,NaN,NaN,NaN,NaN
2,ITC3,Liguria,2016,44.2,21.8,99.7,12.1,7.0,95.0,0.3,...,NaN,NaN,NaN,NaN,7.2,11.6,NaN,NaN,NaN,NaN



📄 neet_gender_year_panel.csv (297 rows × 4 cols)
   Columns: ['year', 'classe_eta', 'sex_label', 'obs_value']
   Year range: 2010 – 2020


,year,classe_eta,sex_label,obs_value
0,2010,Y15-19,female,63.676313
1,2010,Y15-19,male,73.770719
2,2010,Y15-19,total,137.447000



📄 education_expenditure_state_parents_gdp.csv (39 rows × 15 cols)
   Columns: ['REF_AREA', 'Country', 'TIME_PERIOD', 'state_pct_gdp', 'parents_private_pct_gdp', 'rest_world_pct_gdp', 'total_pct_gdp', 'state_usd_ppp', 'parents_private_usd_ppp', 'rest_world_usd_ppp', 'total_usd_ppp', 'state_share_of_total_pct', 'parents_private_share_of_total_pct', 'rest_world_share_of_total_pct', 'implied_gdp_usd_ppp']


...

,school_type,academic_year,codice_istituto,rows,avg_score,min_score,low_score_rows,weak_score_rows,proxy_rows,keyword_hits,proxy_rate,school_proxy_flag
0,paritaria,2015/16,AV1E007009,4,4.25,2,1,1,3,4,0.75,True
1,paritaria,2015/16,FI1E00700X,4,6.50,6,0,0,3,3,0.75,True
2,paritaria,2015/16,GEPS035006,4,3.50,2,1,1,3,4,0.75,True



📄 global_italy_position_oecd_wb_latest.csv (252 rows × 54 cols)
   Columns: ['iso3', 'education_spending_pct_gdp_year', 'education_spending_pct_gdp', 'tertiary_enrollment_gross_pct_year', 'tertiary_enrollment_gross_pct', 'learning_poverty_pct_year', 'learning_poverty_pct', 'access_minus_learning_gap_year', 'access_minus_learning_gap', 'cost_intensity_x_access_year', 'cost_intensity_x_access', 'country', 'oecd_funding_year', 'state_pct_gdp_oecd', 'private_pct_gdp_oecd', 'total_pct_gdp_oecd', 'oecd_funding_year_usd', 'state_usd_ppp_oecd', 'private_usd_ppp_oecd', 'total_usd_ppp_oecd', 'state_share_pct_oecd', 'private_share_pct_oecd', 'per_student_usd_ppp_oecd_year', 'per_student_usd_ppp_oecd', 'education_spending_pct_gdp_rank', 'education_spending_pct_gdp_n', 'education_spending_pct_gdp_pct_better', 'tertiary_enrollment_gross_pct_rank', 'tertiary_enrollment_gross_pct_n', 'tertiary_enrollment_gross_pct_pct_better', 'learning_poverty_pct_rank', 'learning_poverty_pct_n', 'learning_poverty_p

,iso3,education_spending_pct_gdp_year,education_spending_pct_gdp,tertiary_enrollment_gross_pct_year,tertiary_enrollment_gross_pct,learning_poverty_pct_year,learning_poverty_pct,access_minus_learning_gap_year,access_minus_learning_gap,cost_intensity_x_access_year,...,private_pct_gdp_oecd_pct_better,state_share_pct_oecd_rank,state_share_pct_oecd_n,state_share_pct_oecd_pct_better,private_share_pct_oecd_rank,private_share_pct_oecd_n,private_share_pct_oecd_pct_better,per_student_usd_ppp_oecd_rank,per_student_usd_ppp_oecd_n,per_student_usd_ppp_oecd_pct_better
0,ABW,2021.0,3.618558,2024.0,14.065542,NaN,NaN,NaN,NaN,2016.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFE,2023.0,3.962293,2021.0,8.661860,2019.0,90.311594,2019.0,-81.638864,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,2017.0,4.343190,2020.0,10.854360,2013.0,93.267998,NaN,NaN,2014.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



📄 dsu_ersu_support_panel_2024_2025.csv (76 rows × 27 cols)
   Columns: ['region', 'dsu_ente', 'academic_year', 'k', 'entity_is_ersu_like', 'applications_total', 'eligible_students_total', 'beneficiaries_borse_total', 'beneficiaries_prestiti_total', 'beneficiaries_disability_support_total', 'spesa_borse_total', 'spesa_prestiti_total', 'spesa_mobilita_total', 'spesa_disability_support_total', 'spesa_alloggi_total', 'spesa_ristorazione_total', 'spesa_dsu_total', 'posti_alloggio_total', 'posti_alloggio_idonei', 'posti_alloggio_non_idonei', 'posti_mensa_total', 'pasti_erogati_total', 'studenti_mensa_total', 'grant_coverage_rate', 'grant_coverage_rate_capped', 'avg_borsa_support_eur', 'avg_mensa_spend_per_student_eur']


,region,dsu_ente,academic_year,k,entity_is_ersu_like,applications_total,eligible_students_total,beneficiaries_borse_total,beneficiaries_prestiti_total,beneficiaries_disability_support_total,...,posti_alloggio_total,posti_alloggio_idonei,posti_alloggio_non_idonei,posti_mensa_total,pasti_erogati_total,studenti_mensa_total,grant_coverage_rate,grant_coverage_rate_capped,avg_borsa_support_eur,avg_mensa_spend_per_student_eur
0,Abruzzo,ADSU di Chieti,2024-2025,ABRUZZO||ADSU DI CHIETI,True,5680.0,3822.0,9161.0,0.0,41.0,...,163.0,128.0,1.0,0.0,324116.0,6874.0,1.612852,1,3450.827994,0.0
1,Abruzzo,ADSU di L'Aquila,2024-2025,ABRUZZO||ADSU DI L'AQUILA,True,3565.0,2271.0,5690.0,0.0,12.0,...,218.0,215.0,0.0,0.0,187575.0,4865.0,1.596073,1,2888.665786,0.0
2,Abruzzo,ADSU di Teramo,2024-2025,ABRUZZO||ADSU DI TERAMO,True,1089.0,663.0,1464.0,0.0,3.0,...,0.0,0.0,0.0,0.0,25402.0,0.0,1.344353,1,2180.523593,0.0


---
## 4. Existing Notebooks Inventory

What analysis has already been done?

In [18]:
nb_files = list(NOTEBOOKS.glob('*.ipynb'))
display(Markdown(f'### {len(nb_files)} Jupyter Notebooks in `Notebooks/`'))

nb_info = []
for nb in sorted(nb_files):
    with open(nb, 'r', encoding='utf-8') as f:
        content = json.load(f)
    cells = content.get('cells', [])
    code_cells = [c for c in cells if c['cell_type'] == 'code']
    md_cells = [c for c in cells if c['cell_type'] == 'markdown']
    # Extract first markdown heading
    title = '(no title)'
    for mc in md_cells:
        src = ''.join(mc['source'])
        for line in src.split('\n'):
            if line.strip().startswith('# '):
                title = line.strip()[2:].strip()
                break
        if title != '(no title)': break
    nb_info.append({
        'notebook': nb.name,
        'title': title[:80],
        'code_cells': len(code_cells),
        'markdown_cells': len(md_cells),
        'total_cells': len(cells),
        'size': sizeof_fmt(nb.stat().st_size),
    })

nb_df = pd.DataFrame(nb_info)
display(nb_df)

### 16 Jupyter Notebooks in `Notebooks/`

,notebook,title,code_cells,markdown_cells,total_cells,size
0,data_inventory_comprehensive.ipynb,📊 Italienation – Comprehensive Data Inventory,26,23,49,247.1 KB
1,education_spending_outcomes.ipynb,"Education Spending, Household Costs, and Outcomes in Italy, UK, Germany, and EU",10,11,21,196.8 KB
2,italy_bocciatura_repeaters_full_analysis_v2.ipynb,Italy Upper-Secondary Repetition and Bocciatura Pressure,11,11,22,725.4 KB
3,italy_capital_formation_h_c_i.ipynb,"Italy: Human, Cultural, and Intellectual Capital",6,6,12,9.1 KB
4,italy_full_fiscal_landscape.ipynb,Italy Education Fiscal Landscape,9,8,17,439.2 KB
5,italy_human_capital_political_aspects.ipynb,"Italy: Human Capital, Migration, and Political Trust",4,5,9,5.0 KB
6,italy_lower_secondary_middle_school_analysis.ipynb,Italy Lower-Secondary School Analysis (`Scuola media`),13,13,26,24.4 KB
7,italy_middle_to_upper_transition_analysis.ipynb,Italy Middle-to-Upper Transition: Integrated Analysis,14,10,24,27.7 KB
8,italy_neet_full_analysis.ipynb,"Italy's NEET Crisis — Education Barriers, Economic Decay & the Case for Open Acc",31,10,41,112.8 KB
9,italy_oecd_triangle_mobility_analysis.ipynb,Italy Social Mobility Triangle (OECD-style),8,5,13,8.0 KB


In [19]:
# Map notebooks to data domains
display(Markdown('### Notebook → Data Domain Mapping'))
notebook_domains = {
    'education_spending_outcomes.ipynb': ['OECD education spending', 'Eurostat expenditure', 'World Bank education'],
    'italy_bocciatura_repeaters_full_analysis_v2.ipynb': ['ISTAT repeaters upper secondary', 'SNV esiti proxy'],
    'italy_capital_formation_h_c_i.ipynb': ['World Bank HCI', 'OECD capital formation'],
    'italy_full_fiscal_landscape.ipynb': ['SIOPE school budgets', 'Education fiscal inventory', 'MEF IRPEF'],
    'italy_human_capital_political_aspects.ipynb': ['Policy benchmarks', 'Institutional context'],
    'italy_lower_secondary_middle_school_analysis.ipynb': ['ISTAT lower secondary indicators', 'Exam proxy'],
    'italy_middle_to_upper_transition_analysis.ipynb': ['Transition bridge panel', 'ANPAL replacement'],
    'italy_neet_full_analysis.ipynb': ['ISTAT NEET', 'Eurostat NEET', 'Regional model panel'],
    'italy_oecd_triangle_mobility_analysis.ipynb': ['OECD triangle dataset', 'Education-mobility nexus'],
    'italy_oed_goldthorpe_mobility_analysis.ipynb': ['OED components', 'Social mobility'],
    'italy_textbooks_schools_territory.ipynb': ['MinIstruzione textbooks', 'School registry', 'Territory analysis'],
    'italy_tripartite_school_system.ipynb': ['School-track structure', 'Eurostat VET'],
    'neet_italy_analysis.ipynb': ['ISTAT NEET core', 'Basic demographics'],
    'siope_minister_data_exploration.ipynb': ['SIOPE raw treasury', 'MinIstruzione budgets'],
    'territorial_expenditure_analysis.ipynb': ['Territorial spending proxy', 'Regional variation'],
}
for nb_name, domains in notebook_domains.items():
    print(f'📓 {nb_name}')
    for d in domains:
        print(f'   └─ {d}')
    print()

### Notebook → Data Domain Mapping

📓 education_spending_outcomes.ipynb
   └─ OECD education spending
   └─ Eurostat expenditure
   └─ World Bank education

📓 italy_bocciatura_repeaters_full_analysis_v2.ipynb
   └─ ISTAT repeaters upper secondary
   └─ SNV esiti proxy

📓 italy_capital_formation_h_c_i.ipynb
   └─ World Bank HCI
   └─ OECD capital formation

📓 italy_full_fiscal_landscape.ipynb
   └─ SIOPE school budgets
   └─ Education fiscal inventory
   └─ MEF IRPEF

📓 italy_human_capital_political_aspects.ipynb
   └─ Policy benchmarks
   └─ Institutional context

📓 italy_lower_secondary_middle_school_analysis.ipynb
   └─ ISTAT lower secondary indicators
   └─ Exam proxy

📓 italy_middle_to_upper_transition_analysis.ipynb
   └─ Transition bridge panel
   └─ ANPAL replacement

📓 italy_neet_full_analysis.ipynb
   └─ ISTAT NEET
   └─ Eurostat NEET
   └─ Regional model panel

📓 italy_oecd_triangle_mobility_analysis.ipynb
   └─ OECD triangle dataset
   └─ Education-mobility nexus

📓 italy_oed_goldthorpe_mobility_analysis.ipynb

---
## 5. Data Quality Assessment

Check key CSVs for completeness, year coverage, and missing values.

In [20]:
# Check critical datasets for quality
critical_files = {
    'ISTAT NEET (main)': LOCAL / 'ISTAT' / 'istat_neet_new.csv',
    'ISTAT Labour Force': LOCAL / 'ISTAT' / 'istat_labour_force.csv',
    'ISTAT Early School Leavers': LOCAL / 'ISTAT' / 'istat_early_school_leavers.csv',
    'ISTAT Poverty Absolute Inc.': LOCAL / 'ISTAT' / 'istat_poverty_absolute_incidence.csv',
    'Eurostat NEET Detail': LOCAL / 'eurostat' / 'estat_neet_detail.csv',
    'Eurostat Education Spending': LOCAL / 'eurostat' / 'estat_educ_expenditure_pct_gdp.csv',
    'OECD Education Fin GDP': LOCAL / 'oecd' / 'oecd_education_fin_gdp.csv',
    'WB Education Spending': LOCAL / 'worldbank' / 'wb_education_spending_pct_gdp.csv',
    'OECD PISA Trend (Italy)': LOCAL / 'oecd' / 'oecd_it_pisa_trend.csv',
    'WB Gini Index': ROOT / 'local_data' / 'WB_WDI_SI_POV_GINI.csv',
}

quality_rows = []
for label, fpath in critical_files.items():
    if not fpath.exists():
        quality_rows.append({'dataset': label, 'status': '❌ MISSING', 'rows': 0, 'cols': 0, 'null_pct': '-', 'year_range': '-'})
        continue
    try:
        df = pd.read_csv(fpath, low_memory=False)
        null_pct = round(df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100, 1)
        yr_cols = [c for c in df.columns if c.lower() in ('year', 'anno', 'time_period', 'time')]
        yr_range = '-'
        if yr_cols:
            vals = pd.to_numeric(df[yr_cols[0]], errors='coerce').dropna()
            if len(vals):
                yr_range = f"{int(vals.min())}–{int(vals.max())}"
        quality_rows.append({
            'dataset': label,
            'status': '✅ OK',
            'rows': len(df),
            'cols': len(df.columns),
            'null_pct': f'{null_pct}%',
            'year_range': yr_range,
        })
    except Exception as e:
        quality_rows.append({'dataset': label, 'status': f'⚠️ Error: {str(e)[:50]}', 'rows': 0, 'cols': 0, 'null_pct': '-', 'year_range': '-'})

display(Markdown('### Critical Dataset Quality'))
display(pd.DataFrame(quality_rows))

### Critical Dataset Quality

,dataset,status,rows,cols,null_pct,year_range
0,ISTAT NEET (main),✅ OK,57397,13,0.0%,2010–2020
1,ISTAT Labour Force,✅ OK,49485,10,0.0%,2010–2024
2,ISTAT Early School Leavers,✅ OK,2309,18,50.0%,2010–2020
3,ISTAT Poverty Absolute Inc.,✅ OK,157,14,0.0%,2010–2013
4,Eurostat NEET Detail,✅ OK,116160,13,13.3%,2005–2025
5,Eurostat Education Spending,✅ OK,7035,10,19.1%,2012–2023
6,OECD Education Fin GDP,✅ OK,1244,15,8.1%,2015–2020
7,WB Education Spending,✅ OK,17556,8,33.2%,-
8,OECD PISA Trend (Italy),✅ OK,8,10,10.0%,2000–2022
9,WB Gini Index,✅ OK,2351,45,2.2%,1963–2024


---
## 6. SIOPE Data Quality Deep-Dive

Check which years have actual data vs header-only files.

In [21]:
siope_dir = LOCAL / 'SIOPE'
siope_years = []
for f in sorted(siope_dir.glob('siope_uscite_*.csv')):
    size = f.stat().st_size
    try:
        df = pd.read_csv(f, nrows=5)
        nrows = sum(1 for _ in open(f, encoding='utf-8', errors='ignore')) - 1
    except:
        nrows = 0
    year = f.stem.replace('siope_uscite_', '')
    siope_years.append({
        'year': year,
        'rows': nrows,
        'size': sizeof_fmt(size),
        'status': '✅ Data' if nrows > 0 else '⚠️ Empty/header-only',
    })

display(Markdown('### SIOPE Expenditure Files by Year'))
display(pd.DataFrame(siope_years))

### SIOPE Expenditure Files by Year

,year,rows,size,status
0,2020,10250,386.7 KB,✅ Data
1,2021,30715,1.1 MB,✅ Data
2,2022,33856,1.2 MB,✅ Data
3,2023,68742,2.5 MB,✅ Data
4,2024,204499,7.4 MB,✅ Data
5,2025,381038,13.8 MB,✅ Data
6,2026,109041,4.0 MB,✅ Data


---
## 7. Manual-Required Data Tracker

Datasets that could not be fetched programmatically and require manual download.

In [22]:
manual_dir = LOCAL / 'manual_required'
manual_items = []

for f in sorted(manual_dir.glob('*.md')):
    with open(f, encoding='utf-8') as fh:
        content = fh.read()
    manual_items.append({'file': f.name, 'content_preview': content[:200].replace('\n', ' ')})

display(Markdown(f'### {len(list(manual_dir.glob("*")))} files in `manual_required/`'))
for item in manual_items:
    print(f'\n📋 {item["file"]}')
    print(f'   {item["content_preview"]}...')

# Also load the master manifest
manifest_path = manual_dir / 'manual_required_master_manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        master = json.load(f)
    print(f'\n\n🗂️ Master manifest has {len(master) if isinstance(master, list) else "dict"} entries')
    if isinstance(master, list):
        for entry in master[:5]:
            print(f'  • {entry.get("id", entry.get("name", "?"))}: {entry.get("status", "?")}')
    elif isinstance(master, dict):
        for k, v in list(master.items())[:5]:
            print(f'  • {k}: {v.get("status", v) if isinstance(v, dict) else v}')

### 17 files in `manual_required/`


📋 anpal_cpi_data_manual_followup.md
   # ANPAL/CPI Data Manual Follow-up  Auto-generated endpoint checks:  - URL: https://dati.anpal.gov.it/api/3/action/package_list   - status_code: None   - ok: False   - content_type:    - sample/error: ...

📋 inapp_internship_stage_data_manual_required.md
   # INAPP/Isfol Internship (Tirocinio) Prevalence Data  **Priority:** low  ## Why this is needed Italy has a well-documented problem of unpaid or low-paid internships ('stage') as a barrier to youth lab...

📋 inps_apprenticeship_contracts_manual_required.md
   # INPS Apprenticeship Contracts by Region/Sector/Year  **Priority:** high  ## Why this is needed Apprenticeship (contratto di apprendistato) is the main institutionalised school-to-work bridge in Ital...

📋 istat_parental_education_by_region_manual_required.md
   # ISTAT Parental Education Attainment by Region (Intergenerational)  **Priority:** medium  ## Why this is needed Direct parental education attainment distribution by region is need

---
## 8. 🔴 GAP ANALYSIS – What's Missing?

Based on the full inventory, here is a systematic assessment of data gaps.

In [23]:
display(Markdown('''
### 8.1 Critical Gaps (High Priority)

| # | Gap | Impact | Status | Suggested Action |
|---|-----|--------|--------|------------------|
| 1 | **ANPAL Garanzia Giovani** (program-level data) | Cannot measure Youth Guarantee program effectiveness | ⚠️ Replaced with Eurostat/ISTAT proxies | Accept proxy or request ANPAL FOIA |
| 2 | **OECD PISA micro-data** (student-level) | Only have Italy trend summary (oecd_it_pisa_trend.csv = 1.2KB) | 🔴 Missing | Manual download from OECD PISA database |
| 3 | **OECD TALIS** (teacher survey, Italy) | Teacher quality/practice indicators unavailable locally | 🔴 Missing | Download from TALIS 2024 database |
| 4 | **OECD PIAAC** (adult skills) | Cannot assess adult literacy/numeracy pipeline | 🔴 Missing | Manual download from PIAAC portal |
| 5 | **Eurostat ALMP spending** (Active Labour Market Policies) | Cannot benchmark Italy's activation spending | 🔴 Eurostat API 404 | Try alternative Eurostat tables or OECD ALMP |
| 6 | **SIOPE 2014-2019** | Treasury flow files are empty (59 bytes each) | 🔴 Empty files | Re-fetch from SIOPE+ API for those years |
| 7 | **INPS apprenticeship** (detailed by region/year) | Only catalog metadata available, not full data | ⚠️ Partial | Download pinned CSV resources from INPS |

### 8.2 Moderate Gaps

| # | Gap | Impact | Status | Suggested Action |
|---|-----|--------|--------|------------------|
| 8 | **ISTAT parental education by region** | Cannot fully model intergenerational mobility | ⚠️ Proxy only | Fetch ISTAT SDMX flow 172_931_DF_DCCV_NEET1_4 |
| 9 | **INAPP internship/stage data** | Missing non-apprenticeship work experience data | 🔴 Missing | Manual download from INAPP portal |
| 10 | **Eurostat fertility by education** | Only have fertility by age/birth order | ⚠️ Proxy | Not critical; age-based proxy may suffice |
| 11 | **AlmaLaurea occupational outcomes** | Directory exists but content not inspected | ⚠️ Unverified | Check if CSV/data present in subdirectory |
| 12 | **INVALSI raw report ZIP** (39MB) | Compressed – needs extraction to be usable | ⚠️ Needs extraction | Unzip punteggi report_generale_agg_2025.zip |

### 8.3 Nice-to-Have Gaps

| # | Gap | Impact | Status | Suggested Action |
|---|-----|--------|--------|------------------|
| 13 | **OpenCoesione structural projects** | Directory exists but not populated/checked | ⚠️ Unverified | Inspect structural_projects subdirectory |
| 14 | **ISTAT non-observed economy** | PDF + Excel only, no CSV panel | ⚠️ Binary only | Extract tables from XLSX into CSV |
| 15 | **Eurostat NEET sub-datasets** | EurostatNeet/ subdirectory not inspected | ⚠️ Unverified | Check for duplicates vs main estat_ files |
| 16 | **MUR AFAM** (art/music academies) | Small niche sector | ⚠️ Low priority | Available under MUR/iscritti-afam etc. |
| 17 | **Global tuition fee comparison** | Sources doc exists but no structured data | ⚠️ Schema only | Needs manual data entry from sources |

'''))


### 8.1 Critical Gaps (High Priority)

| # | Gap | Impact | Status | Suggested Action |
|---|-----|--------|--------|------------------|
| 1 | **ANPAL Garanzia Giovani** (program-level data) | Cannot measure Youth Guarantee program effectiveness | ⚠️ Replaced with Eurostat/ISTAT proxies | Accept proxy or request ANPAL FOIA |
| 2 | **OECD PISA micro-data** (student-level) | Only have Italy trend summary (oecd_it_pisa_trend.csv = 1.2KB) | 🔴 Missing | Manual download from OECD PISA database |
| 3 | **OECD TALIS** (teacher survey, Italy) | Teacher quality/practice indicators unavailable locally | 🔴 Missing | Download from TALIS 2024 database |
| 4 | **OECD PIAAC** (adult skills) | Cannot assess adult literacy/numeracy pipeline | 🔴 Missing | Manual download from PIAAC portal |
| 5 | **Eurostat ALMP spending** (Active Labour Market Policies) | Cannot benchmark Italy's activation spending | 🔴 Eurostat API 404 | Try alternative Eurostat tables or OECD ALMP |
| 6 | **SIOPE 2014-2019** | Treasury flow files are empty (59 bytes each) | 🔴 Empty files | Re-fetch from SIOPE+ API for those years |
| 7 | **INPS apprenticeship** (detailed by region/year) | Only catalog metadata available, not full data | ⚠️ Partial | Download pinned CSV resources from INPS |

### 8.2 Moderate Gaps

| # | Gap | Impact | Status | Suggested Action |
|---|-----|--------|--------|------------------|
| 8 | **ISTAT parental education by region** | Cannot fully model intergenerational mobility | ⚠️ Proxy only | Fetch ISTAT SDMX flow 172_931_DF_DCCV_NEET1_4 |
| 9 | **INAPP internship/stage data** | Missing non-apprenticeship work experience data | 🔴 Missing | Manual download from INAPP portal |
| 10 | **Eurostat fertility by education** | Only have fertility by age/birth order | ⚠️ Proxy | Not critical; age-based proxy may suffice |
| 11 | **AlmaLaurea occupational outcomes** | Directory exists but content not inspected | ⚠️ Unverified | Check if CSV/data present in subdirectory |
| 12 | **INVALSI raw report ZIP** (39MB) | Compressed – needs extraction to be usable | ⚠️ Needs extraction | Unzip punteggi report_generale_agg_2025.zip |

### 8.3 Nice-to-Have Gaps

| # | Gap | Impact | Status | Suggested Action |
|---|-----|--------|--------|------------------|
| 13 | **OpenCoesione structural projects** | Directory exists but not populated/checked | ⚠️ Unverified | Inspect structural_projects subdirectory |
| 14 | **ISTAT non-observed economy** | PDF + Excel only, no CSV panel | ⚠️ Binary only | Extract tables from XLSX into CSV |
| 15 | **Eurostat NEET sub-datasets** | EurostatNeet/ subdirectory not inspected | ⚠️ Unverified | Check for duplicates vs main estat_ files |
| 16 | **MUR AFAM** (art/music academies) | Small niche sector | ⚠️ Low priority | Available under MUR/iscritti-afam etc. |
| 17 | **Global tuition fee comparison** | Sources doc exists but no structured data | ⚠️ Schema only | Needs manual data entry from sources |



---
## 9. Data Coverage Matrix

Summary of which analytical dimensions are covered.

In [24]:
coverage = [
    {'Domain': 'NEET rates & demographics', 'ISTAT': '✅ Rich', 'Eurostat': '✅ Rich', 'OECD': '✅', 'WB': '—', 'Other': 'UK SDG', 'Processed': '✅ Panels', 'Notebooks': '3'},
    {'Domain': 'Youth unemployment', 'ISTAT': '✅', 'Eurostat': '✅', 'OECD': '—', 'WB': '—', 'Other': '—', 'Processed': '✅', 'Notebooks': '2'},
    {'Domain': 'Education spending (% GDP)', 'ISTAT': '—', 'Eurostat': '✅', 'OECD': '✅', 'WB': '✅', 'Other': 'OWID', 'Processed': '✅ History', 'Notebooks': '2'},
    {'Domain': 'School-level budgets (SIOPE)', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': 'SIOPE', 'Processed': '✅ Summary', 'Notebooks': '2'},
    {'Domain': 'INVALSI test scores', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': 'INVALSI', 'Processed': '—', 'Notebooks': '0'},
    {'Domain': 'PISA scores', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '⚠️ Tiny', 'WB': '—', 'Other': '—', 'Processed': '—', 'Notebooks': '0'},
    {'Domain': 'School outcomes (repeaters)', 'ISTAT': '✅', 'Eurostat': '✅', 'OECD': '—', 'WB': '—', 'Other': 'SNV esiti', 'Processed': '✅', 'Notebooks': '2'},
    {'Domain': 'Early school leaving (ESL)', 'ISTAT': '✅', 'Eurostat': '✅', 'OECD': '—', 'WB': '—', 'Other': '—', 'Processed': '✅', 'Notebooks': '1'},
    {'Domain': 'Tertiary education', 'ISTAT': '✅', 'Eurostat': '✅', 'OECD': '—', 'WB': '✅', 'Other': 'MUR', 'Processed': '✅', 'Notebooks': '1'},
    {'Domain': 'Graduate employment', 'ISTAT': '✅ Rich', 'Eurostat': '—', 'OECD': '✅ Trans.', 'WB': '—', 'Other': 'AlmaLaurea', 'Processed': '⚠️', 'Notebooks': '1'},
    {'Domain': 'Poverty (absolute/relative)', 'ISTAT': '✅ Rich', 'Eurostat': '✅', 'OECD': '—', 'WB': '✅ Gini', 'Other': '—', 'Processed': '—', 'Notebooks': '0'},
    {'Domain': 'Household spending', 'ISTAT': '✅ Rich', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': '—', 'Processed': '✅', 'Notebooks': '1'},
    {'Domain': 'VET (vocational education)', 'ISTAT': '—', 'Eurostat': '✅', 'OECD': '✅', 'WB': '—', 'Other': '—', 'Processed': '—', 'Notebooks': '1'},
    {'Domain': 'Textbooks & school costs', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': 'MinIstr.', 'Processed': '—', 'Notebooks': '1'},
    {'Domain': 'Teacher quality (TALIS)', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '🔴 Missing', 'WB': '✅ trained', 'Other': '—', 'Processed': '—', 'Notebooks': '0'},
    {'Domain': 'School buildings', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': 'MinIstr. 9 files', 'Processed': '—', 'Notebooks': '0'},
    {'Domain': 'Disability/inclusion', 'ISTAT': '✅ xlsx', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': '—', 'Processed': '✅ BES', 'Notebooks': '0'},
    {'Domain': 'Mental health/wellbeing', 'ISTAT': '✅ xlsx', 'Eurostat': '✅', 'OECD': '—', 'WB': '✅ suicide', 'Other': '—', 'Processed': '—', 'Notebooks': '0'},
    {'Domain': 'Social mobility', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '⚠️ proxy', 'WB': '—', 'Other': '—', 'Processed': '✅ OED', 'Notebooks': '2'},
    {'Domain': 'Fiscal (IRPEF/tax)', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': 'MEF', 'Processed': '—', 'Notebooks': '1'},
    {'Domain': 'DSU right-to-study', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': 'MUR DSU', 'Processed': '✅', 'Notebooks': '0'},
    {'Domain': 'University tuition', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '✅ costs', 'WB': '—', 'Other': 'MUR contr.', 'Processed': '✅', 'Notebooks': '1'},
    {'Domain': 'Apprenticeship/ALMP', 'ISTAT': '—', 'Eurostat': '🔴 API 404', 'OECD': '—', 'WB': '—', 'Other': 'INPS partial', 'Processed': '⚠️ Catalog', 'Notebooks': '0'},
    {'Domain': 'Corruption (CPI)', 'ISTAT': '—', 'Eurostat': '—', 'OECD': '—', 'WB': '—', 'Other': 'OWID/TI', 'Processed': '—', 'Notebooks': '0*'},
    {'Domain': 'Regional GDP', 'ISTAT': '—', 'Eurostat': '✅', 'OECD': '—', 'WB': '—', 'Other': '—', 'Processed': '—', 'Notebooks': '0'},
]

coverage_df = pd.DataFrame(coverage)
display(Markdown('### Full Data Coverage Matrix'))
display(coverage_df.style.applymap(
    lambda v: 'background-color: #d4edda' if '✅' in str(v) else 
              ('background-color: #f8d7da' if '🔴' in str(v) else
               ('background-color: #fff3cd' if '⚠️' in str(v) else '')),
    subset=['ISTAT', 'Eurostat', 'OECD', 'WB', 'Other', 'Processed']
))

### Full Data Coverage Matrix

C:\Users\Dell\AppData\Local\Temp\ipykernel_17276\2015350883.py:31: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(coverage_df.style.applymap(


,Domain,ISTAT,Eurostat,OECD,WB,Other,Processed,Notebooks
0,NEET rates & demographics,✅ Rich,✅ Rich,✅,—,UK SDG,✅ Panels,3
1,Youth unemployment,✅,✅,—,—,—,✅,2
2,Education spending (% GDP),—,✅,✅,✅,OWID,✅ History,2
3,School-level budgets (SIOPE),—,—,—,—,SIOPE,✅ Summary,2
4,INVALSI test scores,—,—,—,—,INVALSI,—,0
5,PISA scores,—,—,⚠️ Tiny,—,—,—,0
6,School outcomes (repeaters),✅,✅,—,—,SNV esiti,✅,2
7,Early school leaving (ESL),✅,✅,—,—,—,✅,1
8,Tertiary education,✅,✅,—,✅,MUR,✅,1
9,Graduate employment,✅ Rich,—,✅ Trans.,—,AlmaLaurea,⚠️,1


---
## 10. Notebook Coverage vs Data Domains

Mapping which data sources are used by which notebooks, and identifying **unused data**.

In [25]:
display(Markdown('''
### Data That Has NO Dedicated Notebook Analysis

The following datasets are collected but **not yet analyzed** in any notebook:

| Dataset | Files | Potential Analysis |
|---------|-------|-------------------|
| **INVALSI scores** (48 files, ~40MB compressed) | local_data/INVALSI/ | Regional learning gaps, North-South divide, implicit dropout, digital literacy |
| **MinIstruzione school buildings** (9 files, 37MB) | local_data/MinIstruzione/Edifici/ | Infrastructure quality by region, age of buildings, connectivity |
| **MinIstruzione students** (8 files, 38MB) | local_data/MinIstruzione/Alunni/ | Enrollment patterns, foreign students, grade distribution |
| **MinIstruzione teachers** (1 file, 200KB) | local_data/MinIstruzione/Docenti/ | Teacher distribution, qualifications |
| **MUR DSU right-to-study** (7 files) | local_data/MUR/2025-diritto-allo-studio/ | Scholarship coverage, housing, meals by region |
| **MUR enrollment** (18 files, 56MB) | local_data/MUR/immatricolati/ | University access patterns, mobility, field choice |
| **MUR graduates** (18 files, 74MB) | local_data/MUR/laureati/ | Graduation trends, field distribution, international students |
| **MEF IRPEF fiscal** (7+ files) | local_data/MEF/ | Income distribution, tax burden by region |
| **Poverty indicators** (8 ISTAT files) | local_data/ISTAT/istat_poverty_*.csv | Poverty trends, regional disparities, social exclusion |
| **Eurostat wellbeing/health** (5+ files, 130MB) | local_data/eurostat/ | Self-reported health, life satisfaction, social support |
| **Eurostat labour quality** (5+ files) | local_data/eurostat/ | Part-time, overqualification, job vacancies, long-term unemployment |
| **Eurostat housing/homeownership** (17GB) | local_data/eurostat/ | Housing costs, ownership by age group |
| **UK SDG indicators** (48 files) | local_data/UKSDGstats/ | Cross-country SDG benchmarking |
| **Our World in Data** (14 topics) | local_data/ourWorldData/ | Long-run comparative indicators |
| **Disability/BES inclusion** | local_data/processed/ | Regional inclusion gaps, support adequacy |
| **ISTAT non-observed economy** | local_data/ISTAT/non_observed_economy/ | Shadow economy, irregular work |

'''))


### Data That Has NO Dedicated Notebook Analysis

The following datasets are collected but **not yet analyzed** in any notebook:

| Dataset | Files | Potential Analysis |
|---------|-------|-------------------|
| **INVALSI scores** (48 files, ~40MB compressed) | local_data/INVALSI/ | Regional learning gaps, North-South divide, implicit dropout, digital literacy |
| **MinIstruzione school buildings** (9 files, 37MB) | local_data/MinIstruzione/Edifici/ | Infrastructure quality by region, age of buildings, connectivity |
| **MinIstruzione students** (8 files, 38MB) | local_data/MinIstruzione/Alunni/ | Enrollment patterns, foreign students, grade distribution |
| **MinIstruzione teachers** (1 file, 200KB) | local_data/MinIstruzione/Docenti/ | Teacher distribution, qualifications |
| **MUR DSU right-to-study** (7 files) | local_data/MUR/2025-diritto-allo-studio/ | Scholarship coverage, housing, meals by region |
| **MUR enrollment** (18 files, 56MB) | local_data/MUR/immatricolati/ | University access patterns, mobility, field choice |
| **MUR graduates** (18 files, 74MB) | local_data/MUR/laureati/ | Graduation trends, field distribution, international students |
| **MEF IRPEF fiscal** (7+ files) | local_data/MEF/ | Income distribution, tax burden by region |
| **Poverty indicators** (8 ISTAT files) | local_data/ISTAT/istat_poverty_*.csv | Poverty trends, regional disparities, social exclusion |
| **Eurostat wellbeing/health** (5+ files, 130MB) | local_data/eurostat/ | Self-reported health, life satisfaction, social support |
| **Eurostat labour quality** (5+ files) | local_data/eurostat/ | Part-time, overqualification, job vacancies, long-term unemployment |
| **Eurostat housing/homeownership** (17GB) | local_data/eurostat/ | Housing costs, ownership by age group |
| **UK SDG indicators** (48 files) | local_data/UKSDGstats/ | Cross-country SDG benchmarking |
| **Our World in Data** (14 topics) | local_data/ourWorldData/ | Long-run comparative indicators |
| **Disability/BES inclusion** | local_data/processed/ | Regional inclusion gaps, support adequacy |
| **ISTAT non-observed economy** | local_data/ISTAT/non_observed_economy/ | Shadow economy, irregular work |



---
## 11. Summary Statistics

In [26]:
display(Markdown(f'''
## 📊 Final Summary

### What We Have

| Metric | Count |
|--------|-------|
| **Total files under local_data/** | {len(all_files)} |
| **Total data size** | {sizeof_fmt(all_files['size_bytes'].sum())} |
| **Source organizations** | 17+ (ISTAT, Eurostat, OECD, World Bank, INVALSI, MinIstruzione, MUR, SIOPE, MEF, INPS, ANPAL, AlmaLaurea, OWID, OpenCoesione, UK SDGs, Save the Children, etc.) |
| **Processed/derived datasets** | {len(proc_files)} files |
| **Jupyter notebooks** | {len(nb_files)} analysis notebooks |
| **Data domains covered** | 25+ thematic areas |
| **Year coverage** | Varies: 2000–2025+ for most series |
| **Geographic granularity** | National + Regional (20 regions) + some provincial/school-level |

### Key Strengths
- ✅ **NEET analysis** is the most complete domain (3 notebooks, rich multi-source panels)
- ✅ **Education spending** well covered across OECD/Eurostat/World Bank
- ✅ **School outcomes** (repeaters, exam proxies) have dedicated analysis
- ✅ **School-level fiscal data** (SIOPE 2020-2026) and MinIstruzione budgets are massive
- ✅ **University data** (MUR) is comprehensive: enrollment, graduates, DSU, tuition
- ✅ **Textbook & territory analysis** is uniquely detailed

### Critical Gaps (Action Required)
1. 🔴 **PISA micro-data** – Only have a 1.2KB trend summary
2. 🔴 **TALIS teacher survey** – URLs verified but data not downloaded
3. 🔴 **PIAAC adult skills** – Not available locally
4. 🔴 **SIOPE 2014-2019** – Files are empty (59 bytes each)
5. 🔴 **ALMP spending** – Eurostat API returned 404
6. ⚠️ **INVALSI data unused** – 48 files collected but no notebook analysis
7. ⚠️ **School buildings data unused** – 9 files, 37MB, no analysis
8. ⚠️ **Poverty indicators unused** – Rich ISTAT data but no dedicated notebook
9. ⚠️ **MUR university data unused** – 130+ MB of enrollment/graduate data not analyzed
10. ⚠️ **Wellbeing/health data unused** – Large Eurostat files collected but dormant
'''))


## 📊 Final Summary

### What We Have

| Metric | Count |
|--------|-------|
| **Total files under local_data/** | 691 |
| **Total data size** | 3.6 GB |
| **Source organizations** | 17+ (ISTAT, Eurostat, OECD, World Bank, INVALSI, MinIstruzione, MUR, SIOPE, MEF, INPS, ANPAL, AlmaLaurea, OWID, OpenCoesione, UK SDGs, Save the Children, etc.) |
| **Processed/derived datasets** | 86 files |
| **Jupyter notebooks** | 16 analysis notebooks |
| **Data domains covered** | 25+ thematic areas |
| **Year coverage** | Varies: 2000–2025+ for most series |
| **Geographic granularity** | National + Regional (20 regions) + some provincial/school-level |

### Key Strengths
- ✅ **NEET analysis** is the most complete domain (3 notebooks, rich multi-source panels)
- ✅ **Education spending** well covered across OECD/Eurostat/World Bank
- ✅ **School outcomes** (repeaters, exam proxies) have dedicated analysis
- ✅ **School-level fiscal data** (SIOPE 2020-2026) and MinIstruzione budgets are massive
- ✅ **University data** (MUR) is comprehensive: enrollment, graduates, DSU, tuition
- ✅ **Textbook & territory analysis** is uniquely detailed

### Critical Gaps (Action Required)
1. 🔴 **PISA micro-data** – Only have a 1.2KB trend summary
2. 🔴 **TALIS teacher survey** – URLs verified but data not downloaded
3. 🔴 **PIAAC adult skills** – Not available locally
4. 🔴 **SIOPE 2014-2019** – Files are empty (59 bytes each)
5. 🔴 **ALMP spending** – Eurostat API returned 404
6. ⚠️ **INVALSI data unused** – 48 files collected but no notebook analysis
7. ⚠️ **School buildings data unused** – 9 files, 37MB, no analysis
8. ⚠️ **Poverty indicators unused** – Rich ISTAT data but no dedicated notebook
9. ⚠️ **MUR university data unused** – 130+ MB of enrollment/graduate data not analyzed
10. ⚠️ **Wellbeing/health data unused** – Large Eurostat files collected but dormant


---
## 12. Recommended Next Steps

### Priority 1 – Close Critical Gaps
1. **Download PISA micro-data** from OECD PISA database (Italy + comparators)
2. **Download TALIS 2024** from verified URLs
3. **Re-fetch SIOPE 2014-2019** from SIOPE+ API or RGS archives
4. **Unzip INVALSI report** (punteggi...report_generale_agg_2025.zip)

### Priority 2 – Activate Unused Data
5. **Create INVALSI analysis notebook** – Regional learning gaps, dispersion, digital competence
6. **Create poverty analysis notebook** – ISTAT poverty series + Eurostat social exclusion
7. **Create MUR university notebook** – Enrollment trends, field distribution, dropout
8. **Create school infrastructure notebook** – MinIstruzione Edifici data

### Priority 3 – Integration
9. **Build master regional panel** joining NEET + poverty + INVALSI + spending + school outcomes
10. **Cross-reference with ItaliaCheAffonda** (Coniglione thesis validation project)